# A1.7 · Identity spoofing and impersonation

**Function A — Securing AI Architectures → CyberTravels' Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.6 · Privilege compromise](https://spbreed.github.io/cyber-commons/lessons/A1.6.html)**.

| | |
|---|---|
| Tools used | SPIFFE/SPIRE |

## What this lesson is

**What it covers.** Have two agents share a credential, then try to work out which one made the call.

**Why a security engineer needs it.** Attribution fails before the incident starts: you cannot say which agent acted, so you cannot revoke one without breaking all of them. The control it builds is: per-workload identity with attestation (A2.1, A2.2) and a lifecycle that can revoke one (A2.5).

This is a **risk** lesson: it shows the failure happening before anything tries to stop it, so the control that follows is answering something you have already watched go wrong.

## 1 · The hook

Four agents share one service account. The audit log answers "what happened" perfectly and cannot answer "which one" at all — and neither can the downstream service that was deciding whether to trust the caller.

> **At CyberTravels.** All four agents share one service account, `cybertravels-svc`. The payments API can see that CyberTravels called it and cannot see which of the four — so the refund and the itinerary lookup are indistinguishable to the thing deciding whether to trust the caller. R11.

## 2 · The framework

```
   agent A --+
   agent B --+---> one service account ---> downstream service
   agent C --+          "svc-automation"          |
   agent D --+                                    v
                                        "who called me?"  -> unanswerable

   the audit log is complete and useless: every row has the same subject
```

**OWASP T9 — Identity Spoofing & Impersonation.**

A1.6 was about an agent holding too much authority. This one is about the
**identity** component being unable to say *which agent* is calling at all.

When several agents share one credential — the same service account, the same
API key baked into the same image — they are, to every downstream system, the
same principal. There is no spoofing step required. Impersonation is the
default state, because there was never a distinction to defeat.

Three consequences follow, and the third is the one that hurts during an
incident:

**Authorization cannot differ.** Every agent gets the union of what any of them
needs, which is A1.6 again, arriving through a different door.

**Attribution is impossible.** "Which agent called this?" has no answer. Not a
hard answer — no answer, because the information was never present.

**Revocation is all-or-nothing.** You have one misbehaving agent and one
credential shared by forty. Rotating it stops the incident and stops the other
thirty-nine, so the decision becomes a business call in the middle of a
response, at whatever hour it is.

In a multi-agent topology this compounds. A peer's message is trusted because it
came from a peer — but if identity cannot distinguish peers, "it came from a
peer" is a claim anyone inside the perimeter can make.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

Three agents, one credential, one incident.

## 4 · The check, as a skill

Three CyberTravels agents share one API key, so the payments API records one caller on every line. The skill measures both costs: what the record can attribute, and what revoking the key would stop.

### The skill — [`skills/threats/shared-credential-attribution-check/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/threats/shared-credential-attribution-check/SKILL.md)

```yaml
name: shared-credential-attribution-check
description: >-
  Find credentials held by more than one agent and show what sharing does to
  attribution and to containment — who the downstream records, and what
  revoking it stops. Use when several agents call the same API, or when asked
  which agent did something and the record cannot say.
allowed-tools: Read, Grep, Glob
```

# One credential, three agents, one line in the log

A shared credential is usually adopted for convenience and paid for during an
incident. It costs two things at once: the downstream cannot attribute an
action to an agent, and the only containment available stops **every** holder.

## When to use this

Whenever more than one agent, job or replica authenticates downstream. Also
before an incident: this is the check whose absence turns a contained problem
into an outage.

## Procedure

**1 — Enumerate holders per credential.** Group by the secret, not by the
service. Environment variables, mounted files, a shared secrets-manager path
and a baked-in image layer are all the same credential when the value matches.

**2 — Read a downstream record.** Whatever the caller is identified by — an API
key id, a client id, a service account — record what the downstream can print.
If three agents map to one identifier, attribution ends there.

**3 — Simulate the destructive call.** Have one holder perform something
irreversible. Ask, from the downstream record alone, which holder did it. Write
down the answer even when it is "cannot be determined"; that sentence is the
finding.

**4 — Cost the containment.** Revoke the credential on paper and list what
stops. The count of unrelated things that stop is the number to report.

**5 — Propose per-workload identity,** and say what it costs: one credential
per agent, issued by the platform rather than pasted, so revocation is
per-agent and attribution is free.

## Output contract

```json
{
  "credentials": [{"id": "str", "holders": ["str"], "source": "env|file|manager|image"}],
  "downstream_identifier": {"field": "str", "distinguishes_holders": false},
  "destructive_probe": {"actor": "str", "recoverable_from_record": false},
  "containment": {"revoking_stops": ["str"], "collateral": 0}
}
```

## Failure modes

- **Grouping by service instead of by secret value.** Two names, one key, is
  still one credential.
- **Assuming the log will disambiguate.** Read an actual row before assuming a
  field exists.
- **Reporting only attribution.** The containment cost is what makes it urgent.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/threats/shared-credential-attribution-check/scripts/shared_credential_attribution_check.py
SCRIPT = "skills/threats/shared-credential-attribution-check/scripts/shared_credential_attribution_check.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Three agents share one credential, so the downstream record shows a single caller on every line. When one deletes a production table the culprit is not recoverable from the record, and the only containment available stops all three.

## Your turn

Count the distinct credentials across your agents and divide by the number of agents. Any answer below one is this risk, and the number tells you how many innocent agents a revocation takes down.

---

**Next → [A1.8 · Malicious code execution](https://spbreed.github.io/cyber-commons/lessons/A1.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*